# Smart Farm Surveillance System - Jupyter Notebook Edition

This notebook runs the YOLOv8-based Smart Farm Surveillance System inline within your Jupyter workspace.

### Instructions:
1. Run the cells in order.
2. To stop the webcam stream, click the **Stop** (Interrupt kernel) button in the Jupyter toolbar.

In [1]:
import cv2
import os
import time
import datetime
import threading
from ultralytics import YOLO
import IPython.display as IPydisp

# ── Twilio SMS (optional) ─────────────────────────────────────────────────────
try:
    from twilio.rest import Client as TwilioClient
    TWILIO_AVAILABLE = True
except ImportError:
    TWILIO_AVAILABLE = False

# ── Raspberry Pi GPIO (optional) ──────────────────────────────────────────────
try:
    import RPi.GPIO as GPIO
    GPIO_AVAILABLE = True
except ImportError:
    GPIO_AVAILABLE = False

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
MODEL_PATH       = "best.pt"          # path to your trained YOLOv8 weights
CAMERA_SOURCE    = 0                   # 0 = webcam, or RTSP stream URL
CONFIDENCE       = 0.50                # minimum detection confidence
ALERT_CLASSES    = ["deer", "wild_boar", "cow", "unknown_person"]
ALERT_COOLDOWN   = 30                  # alert cooldown per class in seconds
SNAPSHOT_DIR     = "detections"

# Twilio settings (leave blank to disable SMS)
TWILIO_SID       = ""
TWILIO_TOKEN     = ""
TWILIO_FROM      = ""
FARMER_PHONE     = ""

# GPIO alarm settings (Raspberry Pi)
ALARM_GPIO_PIN   = 17
ALARM_DURATION   = 5

# Motion filter sensitivity
MOTION_MIN_AREA  = 800

CLASS_COLORS = {
    "deer":           (0,   200,  80),
    "wild_boar":      (0,   140, 255),
    "cow":            (255, 180,   0),
    "unknown_person": (0,    0,  255),
}
DEFAULT_COLOR = (200, 200, 200)

In [2]:
def setup_gpio():
    if GPIO_AVAILABLE and ALARM_GPIO_PIN is not None:
        GPIO.setmode(GPIO.BCM)
        GPIO.setup(ALARM_GPIO_PIN, GPIO.OUT)
        GPIO.output(ALARM_GPIO_PIN, GPIO.LOW)
        print(f"[GPIO] Alarm relay ready on pin {ALARM_GPIO_PIN}")

_alarm_active = False

def trigger_alarm():
    global _alarm_active
    if GPIO_AVAILABLE and ALARM_GPIO_PIN is not None:
        if _alarm_active:
            return
        def run_alarm():
            global _alarm_active
            try:
                _alarm_active = True
                print(f"[GPIO] Alarm activated on pin {ALARM_GPIO_PIN}")
                GPIO.output(ALARM_GPIO_PIN, GPIO.HIGH)
                time.sleep(ALARM_DURATION)
                GPIO.output(ALARM_GPIO_PIN, GPIO.LOW)
                print("[GPIO] Alarm deactivated")
            except Exception as e:
                print(f"[GPIO] Alarm error: {e}")
            finally:
                _alarm_active = False
        thread = threading.Thread(target=run_alarm, daemon=True)
        thread.start()

def send_sms(label, confidence, camera_id=0):
    if not TWILIO_AVAILABLE or not TWILIO_SID:
        return
    try:
        client = TwilioClient(TWILIO_SID, TWILIO_TOKEN)
        ts  = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        msg = (
            f"[FARM ALERT] {ts}\n"
            f"Detected: {label.upper()} ({confidence:.0%})\n"
            f"Camera: #{camera_id}\n"
            f"Check your farm immediately!"
        )
        client.messages.create(body=msg, from_=TWILIO_FROM, to=FARMER_PHONE)
        print(f"[SMS] Alert sent for {label}")
    except Exception as e:
        print(f"[SMS] Failed to send: {e}")

def send_sms_async(label, confidence, camera_id=0):
    thread = threading.Thread(target=send_sms, args=(label, confidence, camera_id), daemon=True)
    thread.start()

def save_snapshot(frame, label, confidence):
    os.makedirs(SNAPSHOT_DIR, exist_ok=True)
    ts   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    name = f"{SNAPSHOT_DIR}/{ts}_{label}_{confidence:.2f}.jpg"
    cv2.imwrite(name, frame)
    print(f"[SNAP] Saved {name}")

class MotionFilter:
    def __init__(self):
        self.subtractor = cv2.createBackgroundSubtractorMOG2(
            history=500, varThreshold=50, detectShadows=False
        )
    def has_motion(self, frame):
        mask      = self.subtractor.apply(frame)
        kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask      = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        return any(cv2.contourArea(c) > MOTION_MIN_AREA for c in contours)

class AlertManager:
    def __init__(self):
        self._last_alert = {}
    def can_alert(self, label):
        now  = time.time()
        last = self._last_alert.get(label, 0)
        if now - last >= ALERT_COOLDOWN:
            self._last_alert[label] = now
            return True
        return False

def draw_box(frame, x1, y1, x2, y2, label, confidence):
    color = CLASS_COLORS.get(label, DEFAULT_COLOR)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    text    = f"{label}  {confidence:.0%}"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 8, y1), color, -1)
    cv2.putText(frame, text, (x1 + 4, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

def draw_status(frame, fps, motion, detections):
    h, w = frame.shape[:2]
    ts   = datetime.datetime.now().strftime("%Y-%m-%d  %H:%M:%S")
    cv2.putText(frame, ts, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)
    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 46), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)
    motion_txt   = "Motion: YES" if motion else "Motion: no"
    motion_color = (0, 255, 100) if motion else (120, 120, 120)
    cv2.putText(frame, motion_txt, (w - 160, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, motion_color, 1)
    if detections:
        alert_txt = f"ALERT: {', '.join(sorted(list(set(detections))))}"
        cv2.putText(frame, alert_txt, (10, h - 14), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────
global MODEL_PATH
if not os.path.exists(MODEL_PATH):
    print(f"[WARNING] Model weights file '{MODEL_PATH}' not found.")
    if MODEL_PATH == "best.pt":
        print("[INFO] Falling back to standard 'yolov8n.pt' for testing...")
        MODEL_PATH = "yolov8n.pt"

print(f"[MODEL] Loading {MODEL_PATH} ...")
model = YOLO(MODEL_PATH)

# ── Open camera ───────────────────────────────────────────────────────────
print(f"[CAM] Opening source: {CAMERA_SOURCE}")
cap = cv2.VideoCapture(CAMERA_SOURCE)
if not cap.isOpened():
    print("[ERROR] Cannot open camera. Check CAMERA_SOURCE.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    setup_gpio()
    motion_filter = MotionFilter()
    alert_manager = AlertManager()

    prev_time  = time.time()
    frame_count = 0

    print("[SYS] Active - Click 'Stop' (Interrupt Kernel) button to exit.\n")

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("[CAM] Frame read failed. Retrying...")
                time.sleep(0.5)
                continue

            # ── FPS ───────────────────────────────────────────────────────────────
            now      = time.time()
            fps      = 1.0 / max(now - prev_time, 1e-6)
            prev_time = now
            frame_count += 1
            motion = motion_filter.has_motion(frame)
            detected_labels = []

            if motion:
                results = model(frame, conf=CONFIDENCE, verbose=False)
                for result in results:
                    for box in result.boxes:
                        cls_id     = int(box.cls[0])
                        label      = model.names[cls_id]
                        confidence = float(box.conf[0])
                        x1, y1, x2, y2 = map(int, box.xyxy[0])

                        draw_box(frame, x1, y1, x2, y2, label, confidence)

                        if label in ALERT_CLASSES:
                            detected_labels.append(label)
                            print(f"[DETECTION] {label.upper()} detected ({confidence:.0%})")

                            if alert_manager.can_alert(label):
                                print(f"[ALERT SMS/ALARM] Sending alerts for {label.upper()}...")
                                save_snapshot(frame, label, confidence)
                                send_sms_async(label, confidence)
                                if label == "unknown_person":
                                    trigger_alarm()

            # ── Overlay status ────────────────────────────────────────────────────
            draw_status(frame, fps, motion, detected_labels)
            
            # ── Display inline in Jupyter ──────────────────────────────────────────
            _, jpeg_data = cv2.imencode('.jpg', frame)
            IPydisp.clear_output(wait=True)
            IPydisp.display(IPydisp.Image(data=jpeg_data.tobytes()))

    except KeyboardInterrupt:
        print("[SYS] Jupyter Kernel Interrupted. Stopping surveillance...")
    finally:
        cap.release()
        if GPIO_AVAILABLE:
            GPIO.cleanup()
        print("[SYS] Camera source released. System stopped.")

[WARNING] Model weights file 'best.pt' not found.
[INFO] Falling back to standard 'yolov8n.pt' for testing...
[MODEL] Loading yolov8n.pt ...
[CAM] Opening source: 0
[SYS] Active - Click 'Stop' (Interrupt Kernel) button to exit.

